# Experimentos TDAH — una corrida

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jpospinalo/tdha-revision/blob/main/tdha_experimentos.ipynb)

Ejecuta **una corrida** —un sitio, un subconjunto de ROIs, un enventanado, una
arquitectura— y sube sus resultados. Cada corrida escribe en su propia carpeta (nombre
derivado de la configuración), así que varias personas pueden correr en paralelo sin
chocar; al final se compilan todas juntas.

## Reglas

1. **No cambiar `SEED`.** El 42 mantiene las mismas particiones en todas las corridas, lo
   que hace la comparación pareada.
2. **Editar solo la celda de configuración.** El resto está en `src/`.
3. **No editar a mano los CSV de `results/`.** Repetir una configuración la reemplaza.
4. Si Colab desconecta, reejecutar el notebook: la corrida incompleta se detecta y se rehace.

## Cuánto tarda

Lo domina el subconjunto de ROIs, que fija el tamaño del modelo:

| ROIs | Características | Parámetros | Duración en GPU |
|---|---|---|---|
| 12 | 66 | 100 mil | decenas de minutos |
| 18 | 153 | 145 mil | decenas de minutos |
| 39 | 741 | 446 mil | ~1 hora |
| 116 | 6.670 | 3,5 millones | varias horas |

Para encadenar varias configuraciones en una sesión: `run_queue.py --in-process` (ver
`docs/performance.md`).

## 1. Preparar el entorno

In [ ]:
# Verificar que hay GPU asignada.
# Si no aparece nada: Entorno de ejecución > Cambiar tipo de entorno > GPU
!nvidia-smi -L || echo "SIN GPU — la corrida será mucho más lenta"

In [ ]:
import os, subprocess, sys, importlib.util

REPO = "https://github.com/jpospinalo/tdha-revision.git"
ROOT = "/content/" + REPO.rstrip("/").split("/")[-1].replace(".git", "")

if os.path.exists(ROOT):
    # Actualizar: si la sesión de Colab se reutiliza, el clon puede tener código
    # viejo y la corrida quedaría marcada con un commit que no es el que se ejecutó.
    print(subprocess.run(["git", "pull", "--ff-only", "origin", "main"], cwd=ROOT,
                         capture_output=True, text=True).stdout.strip())
else:
    subprocess.run(["git", "clone", REPO, ROOT], check=True)

# Instalar SOLO lo que falte. Colab ya trae TensorFlow, NumPy, pandas, scikit-learn,
# SciPy, statsmodels, matplotlib y joblib; un "pip install -r requirements.txt" a
# ciegas puede tardar minutos reinstalando TensorFlow y, peor, dejar una versión
# incompatible con el CUDA del entorno.
faltan = [m for m in ["tensorflow", "keras", "sklearn", "numpy", "pandas",
                      "joblib", "scipy", "statsmodels", "matplotlib"]
          if importlib.util.find_spec(m) is None]
if faltan:
    print("instalando:", faltan)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + faltan, check=True)
else:
    print("todas las dependencias ya están disponibles")

os.chdir(f"{ROOT}/src")
sys.path.insert(0, os.getcwd())
print("listo:", os.getcwd())
print("commit:", subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=ROOT,
                                capture_output=True, text=True).stdout.strip())

## 2. Configuración

**Esta es la única celda que hay que editar.** Se agrupa en tres bloques: qué correr
(modelo e hiperparámetros), la representación de la conectividad, y el enventanado.

| Parámetro | Valores |
|---|---|
| `SITIO` | `NYU`, `Peking`, `NeuroIMAGE`, `OHSU` |
| `ROI_SET` | `12`, `18`, `39`, `116` |
| `MODELO` | `lstm`, `gru`, `cnn1d`, `transformer`, `deepsets`, `brainnetcnn` |
| `HIPERPARAMS` | dict con los del modelo (ver `--list-models`). Ej. `{"units": 64, "dropout": 0.2}` |
| `REPRESENTACION` | `auto`, `ordered`, `static`, `mean`, `mean_std`, `permuted` |

**Representación** — qué recibe el modelo:

- `ordered`: secuencia dinámica de matrices de conectividad, una por ventana.
- `static`: **una sola** matriz de conectividad sobre toda la serie (sin ventanas).
- `mean` / `mean_std`: resumen invariante al orden de las ventanas.
- `permuted`: ventanas barajadas (control del diagnóstico de orden).
- `auto`: dinámica en todos los sitios salvo OHSU, que usa `static`.

`brainnetcnn` opera sobre la **matriz** de conectividad (filtros topológicos que respetan el grafo, no una Conv2D ingenua); úsalo con `REPRESENTACION = "static"`.

**Entrenamiento** (opcional; deja en `None`/`False` para usar los valores por defecto del
script): `LR`, `BATCH_SIZE`, `EPOCHS`, `PATIENCE`, `MIXED_PRECISION` (acelera los modelos
grandes en GPU), `FISHER_Z` (transformación Fisher z de las correlaciones), `TAG` (etiqueta
para distinguir variantes de una misma configuración).

El **enventanado** solo aplica a las representaciones dinámicas; con `static` se ignora.
Admite además ventana `rectangular` o `gaussian` (`WINDOW_SHAPE`).

| Sitio | Escaneo | Ventana | Paso | Solape | Nº ventanas | Rep. por defecto |
|---|---|---|---|---|---|---|
| NYU | 344 s | 120 s | 12 s | 90% | 19 | dinámica |
| Peking | 464 s | 120 s | 18 s | 85% | 20 | dinámica |
| NeuroIMAGE | 504 s | 120 s | 20 s | 84% | 20 | dinámica |
| OHSU | 185 s | 80 s* | 15 s | 81% | 8 | **estática** |

\*OHSU: el escaneo es demasiado corto para una ventana válida (≥ 111 s); su default es
estático. Los 80 s solo aplican si fuerzas una representación dinámica, y es exploratorio.
Recuerda `N_SPLITS` a 5 en OHSU y NeuroIMAGE, y `CLASS_WEIGHT` en Peking.

Para probar configuraciones: cambia lo que quieras en la celda y reejecuta desde ella hacia abajo.
Cualquier parámetro avanzado no listado aquí (p. ej. `--overlap`, `--clipnorm`,
`--deterministic`, o el control anatómico `--random-subset`) se pasa sin editar código con
`ejecutar(["--flag", "valor"])`. La lista completa: `!python run_experiment.py --help`.

### Ejemplos por arquitectura

Copia los valores a los bloques **Qué correr** y **Representación** de la celda de configuración.

**LSTM** (base del artículo) — recurrente sobre la secuencia dinámica:
```python
MODELO = "lstm";  HIPERPARAMS = {};              REPRESENTACION = "ordered"
```
**GRU** — recurrente más frugal, misma entrada:
```python
MODELO = "gru";   HIPERPARAMS = {"units": 128};  REPRESENTACION = "ordered"
```
**CNN1D** — convolución sobre el tiempo (kernel corto = transiciones locales):
```python
MODELO = "cnn1d"; HIPERPARAMS = {"kernel_size": 5}; REPRESENTACION = "ordered"
```
**Transformer como modelo de conjuntos** — sin codificación posicional (apropiado si el orden no discrimina):
```python
MODELO = "transformer"; HIPERPARAMS = {"positional": False}; REPRESENTACION = "ordered"
```
**DeepSets** — invariante al orden por construcción; baseline frente a las recurrentes:
```python
MODELO = "deepsets"; HIPERPARAMS = {"pooling": "mean"}; REPRESENTACION = "ordered"
```
**BrainNetCNN sobre la matriz estática** — convolución topológica sobre una sola matriz de conectividad:
```python
MODELO = "brainnetcnn"; HIPERPARAMS = {}; REPRESENTACION = "static"
```
**Conectividad estática con un modelo vectorial** (una matriz por sujeto, sin ventanas):
```python
MODELO = "deepsets"; REPRESENTACION = "static"
```

Para acelerar los modelos grandes (39/116 ROIs, transformer, brainnetcnn) en GPU: `MIXED_PRECISION = True`.
Los hiperparámetros de cada modelo se listan con la celda `--list-models` de más abajo.

In [ ]:
# ---------------------------------------------------------------------------
# CONFIGURACIÓN DE ESTA CORRIDA   (editar solo esta celda)
# ---------------------------------------------------------------------------

# --- Qué correr ---
SITIO       = "NYU"          # NYU · Peking · NeuroIMAGE · OHSU
ROI_SET     = "12"           # 12 · 18 · 39 · 116
MODELO      = "lstm"         # lstm · gru · cnn1d · transformer · deepsets · brainnetcnn
HIPERPARAMS = {}             # hiperparámetros del modelo (corre la celda --list-models
                             # para ver los de cada uno). Ej.: {"units": 64, "dropout": 0.2}

# --- Representación de la conectividad ---
#   auto · ordered · static · mean · mean_std · permuted   (ver tabla de la celda anterior)
REPRESENTACION = "auto"

# --- Enventanado (solo dinámico; 'static' lo ignora) ---
VENTANA_POR_SITIO = {
    "NYU":        {"window_seconds": 120, "step_seconds": 12},  # 60/6 TR · ov 90% · ~19 ventanas
    "Peking":     {"window_seconds": 120, "step_seconds": 18},  # 60/9 TR · ov 85% · ~20 ventanas
    "NeuroIMAGE": {"window_seconds": 120, "step_seconds": 20},  # 61/10 TR · ov 84% · ~20 ventanas
    "OHSU":       {"window_seconds": 80,  "step_seconds": 15},  # 32/6 TR · exploratorio (< piso 111 s)
}
REP_POR_DEFECTO = {"OHSU": "static"}   # el resto usa 'ordered' cuando REPRESENTACION="auto"
WINDOW_SHAPE   = "rectangular"  # "rectangular" o "gaussian"
GAUSSIAN_SIGMA = None           # sigma en TR de la gaussiana; None = window/6
WINDOW_TR = None    # override avanzado en TR (fija AMBOS); None = usar los segundos de arriba
STEP_TR   = None

# --- Entrenamiento (opcional; None/False = valor por defecto del script) ---
LR              = None    # tasa de aprendizaje (def. 1e-4)
BATCH_SIZE      = None    # tamaño de lote (def. 8)
EPOCHS          = None    # épocas máximas (def. 150)
PATIENCE        = None    # paciencia del early stopping (def. 25)
MIXED_PRECISION = False   # acelera modelos grandes en GPU (T4); cambia los decimales
FISHER_Z        = False   # transformación Fisher z de las correlaciones
TAG             = None    # etiqueta para distinguir variantes de una misma configuración

# --- Validación e identidad ---
N_SPLITS     = 10        # bajar a 5 en NeuroIMAGE y OHSU
N_REPEATS    = 5         # 10 x 5 = 50 evaluaciones externas
CLASS_WEIGHT = False     # activar en Peking
SEED         = 42        # NO CAMBIAR
NOMBRE       = "Juan"        # autor del commit de resultados
CORREO       = "juan@ejemplo.com"
# ---------------------------------------------------------------------------

# Resolución a partir de lo anterior (no hace falta editar debajo de esta línea).
if REPRESENTACION == "auto":
    REPRESENTACION = REP_POR_DEFECTO.get(SITIO, "ordered")
_cfg = VENTANA_POR_SITIO[SITIO]
WINDOW_SECONDS = _cfg["window_seconds"]
STEP_SECONDS   = _cfg["step_seconds"]

if REPRESENTACION == "static":
    _vent = "estática (toda la serie)"
elif WINDOW_TR is not None or STEP_TR is not None:
    _vent = f"{WINDOW_TR}/{STEP_TR} TR"
else:
    _vent = f"{WINDOW_SECONDS}/{STEP_SECONDS} s ({WINDOW_SHAPE})"
print(f"{SITIO} · {ROI_SET} ROIs · {MODELO} {HIPERPARAMS or ''} · "
      f"repr. {REPRESENTACION} · ventana {_vent} · "
      f"{N_SPLITS}x{N_REPEATS} = {N_SPLITS * N_REPEATS} repeticiones")

# La identidad se fija ANTES de correr: run_experiment.py la registra en config.json.
if NOMBRE and CORREO:
    for k, v in [("user.name", NOMBRE), ("user.email", CORREO)]:
        subprocess.run(["git", "config", k, v], cwd=ROOT, check=True)
    print(f"autor de la corrida: {NOMBRE} <{CORREO}>")
else:
    print("\nAVISO: sin NOMBRE ni CORREO la corrida quedará como 'desconocido' y no se "
          "podrá subir.")


In [ ]:
# Subconjuntos de ROIs y arquitecturas disponibles.
!python run_experiment.py --list-roi-sets
!python run_experiment.py --list-models

## 3. Comprobaciones previas

Dos verificaciones antes de gastar GPU. La primera revisa el repositorio y los datos;
la segunda valida esta configuración concreta y sus particiones sin entrenar.

Conviene mirar el número de ventanas, que no aparezcan avisos, y anotar la `huella`:
dos corridas solo son comparables si coinciden su sitio, su semilla y esa huella.

In [ ]:
!python verify_setup.py

In [ ]:
import run_experiment


def ejecutar(extra=()):
    """Ejecuta run_experiment en este proceso con la configuración de la celda anterior.

    Toma todos los parámetros de esa celda (modelo, representación, enventanado y
    entrenamiento). 'extra' añade o sobrescribe argumentos: si aparece dos veces uno,
    gana el último, por eso la prueba de humo puede acortar épocas y pliegues.
    """
    extra = list(extra)
    joined = " ".join(extra)
    es_estatica = REPRESENTACION == "static" or "--representation static" in joined

    # Enventanado: en TR si se fijó el override, en segundos en otro caso.
    if WINDOW_TR is not None or STEP_TR is not None:
        if WINDOW_TR is None or STEP_TR is None:
            raise ValueError("En modo TR hay que fijar WINDOW_TR y STEP_TR juntos.")
        win_args = ["--window", str(WINDOW_TR), "--step", str(STEP_TR)]
    else:
        win_args = ["--window-seconds", str(WINDOW_SECONDS), "--step-seconds", str(STEP_SECONDS)]

    # Representación. El 'extra' tiene prioridad; la estática no lleva enventanado.
    if "--representation" in extra:
        rep_args = []
    elif REPRESENTACION == "static":
        rep_args = ["--representation", "static"]
    elif REPRESENTACION == "ordered":
        rep_args = []
    else:
        rep_args = ["--representation", REPRESENTACION]

    # La forma de ventana y el enventanado solo aplican a las representaciones dinámicas.
    shape_args = []
    if es_estatica:
        win_args = []
    else:
        if WINDOW_SHAPE != "rectangular":
            shape_args = ["--window-shape", WINDOW_SHAPE]
            if GAUSSIAN_SIGMA is not None:
                shape_args += ["--gaussian-sigma", str(GAUSSIAN_SIGMA)]

    argv = ["--site", SITIO, "--roi-set", str(ROI_SET), "--model", MODELO,
            *rep_args, *win_args, *shape_args, "--seed", str(SEED),
            "--n-splits", str(N_SPLITS), "--n-repeats", str(N_REPEATS)]
    if HIPERPARAMS:
        argv += ["--model-arg"] + [f"{k}={v}" for k, v in HIPERPARAMS.items()]

    # Entrenamiento (solo lo que no sea None/False).
    for flag, val in [("--lr", LR), ("--batch-size", BATCH_SIZE),
                      ("--epochs", EPOCHS), ("--patience", PATIENCE), ("--tag", TAG)]:
        if val is not None:
            argv += [flag, str(val)]
    if MIXED_PRECISION:
        argv.append("--mixed-precision")
    if FISHER_Z:
        argv.append("--fisher-z")
    if CLASS_WEIGHT:
        argv.append("--class-weight")

    argv += extra
    print("run_experiment.py " + " ".join(argv) + "\n", flush=True)

    try:
        return run_experiment.main(argv), None
    except SystemExit as e:
        return None, str(e)


_, aviso = ejecutar(["--dry-run"])
if aviso:
    print(aviso)


## 4. Prueba de humo

Dos pliegues y tres épocas, para verificar que el camino completo funciona antes de
lanzar una corrida larga. Escribe en `/tmp`, así que no ensucia los resultados ni se
sube nada.

In [ ]:
_, aviso = ejecutar(["--n-splits", "2", "--n-repeats", "1",
                     "--epochs", "3", "--patience", "2",
                     "--out", "/tmp/smoke", "--tag", "smoke", "--verbose"])
print("\nprueba de humo:", "FALLÓ\n" + aviso if aviso else "correcta")

## 5. La corrida

Aquí ocurre el trabajo. Si Colab desconecta a mitad, se reejecuta el notebook desde el
principio: la carpeta incompleta se detecta y se rehace.

In [ ]:
RUN_ID, aviso = ejecutar(["--verbose"])

# Repetir una configuración reemplaza la anterior: no quedan dos versiones de lo
# mismo, y una corrida interrumpida se rehace sin más.
if aviso:
    raise RuntimeError(aviso)

print("\n" + "=" * 70)
print("identificador de la corrida:", RUN_ID)

## 6. Resultados

`metrics_val.csv` es el archivo principal: una fila por repetición de la validación
cruzada. Los demás permiten reconstruir después las curvas de convergencia, las
matrices de confusión y las curvas ROC sin reentrenar.

`resumen.md` es la vista legible de esta corrida (configuración + métricas titulares), derivada del `config.json`.

In [ ]:
import pandas as pd
from pathlib import Path

RUTA = Path(ROOT) / "results" / "runs" / RUN_ID
val = pd.read_csv(RUTA / "metrics_val.csv")
tra = pd.read_csv(RUTA / "metrics_train.csv")

print(f"{len(val)} repeticiones · época elegida: mediana {val.best_epoch.median():.0f}, "
      f"rango [{val.best_epoch.min():.0f}, {val.best_epoch.max():.0f}]\n")
M = ["accuracy","balanced_accuracy","f1_macro","precision","recall","specificity","auc"]
resumen = pd.DataFrame({
    "entrenamiento": [tra[m].mean() * 100 for m in M],
    "validación":    [val[m].mean() * 100 for m in M],
    "desv. val.":    [val[m].std() * 100 for m in M],
}, index=M).round(2)
resumen["brecha"] = (resumen.entrenamiento - resumen["validación"]).round(2)
print(resumen.to_string())

La mediana de `best_epoch` muestra cuánto cómputo se gasta tras converger: si queda muy
por debajo del tope de épocas, baja `--patience`. Ver `docs/performance.md`.

In [ ]:
print("archivos generados:\n")
for f in sorted(RUTA.glob("*")):
    filas = f"{sum(1 for _ in f.open()) - 1:>6d} filas" if f.suffix == ".csv" else ""
    print(f"  {f.name:26s} {filas}")

In [ ]:
# La dispersión importa tanto como la media: con muestras pequeñas, repeticiones
# individuales muy altas reflejan particiones favorables, no mejor aprendizaje.
import matplotlib.pyplot as plt

fig, (a, b) = plt.subplots(1, 2, figsize=(11, 3.5))
a.hist(val.accuracy * 100, bins=15, edgecolor="white")
a.axvline(val.accuracy.mean() * 100, color="crimson", label="media")
a.set_xlabel("accuracy de validación (%)"); a.set_ylabel("repeticiones"); a.legend()

# Época elegida por pliegue. Si se concentra en valores muy bajos, la partición
# interna es demasiado pequeña para elegir con fiabilidad: ver --start-from-epoch.
b.hist(val.best_epoch, bins=20, edgecolor="white", color="tab:green")
b.axvline(val.best_epoch.median(), color="crimson", label="mediana")
b.set_xlabel("época elegida"); b.set_ylabel("pliegues"); b.legend()
plt.tight_layout(); plt.show()


## 7. Subir los resultados

El push necesita un token de acceso personal de GitHub. **No escribirlo en el
notebook**: se guarda en el panel de secretos de Colab (icono de la llave, a la
izquierda) con el nombre `GITHUB_TOKEN`, y desde ahí se lee sin que quede en el
archivo ni en el historial.

Para crearlo: GitHub → Settings → Developer settings → Personal access tokens →
Fine-grained tokens, con permiso de escritura sobre este repositorio.

In [ ]:
import getpass

if not NOMBRE or not CORREO:
    raise ValueError("Complete NOMBRE y CORREO en la celda de configuración y "
                     "vuelva a ejecutar desde ahí: la identidad debe estar fijada "
                     "ANTES de la corrida para quedar registrada en config.json.")
if not (RUTA / "metrics_val.csv").exists():
    raise FileNotFoundError(f"No hay métricas en {RUTA}. La corrida no terminó.")

try:
    from google.colab import userdata
    TOKEN = userdata.get("GITHUB_TOKEN")
except Exception:
    TOKEN = getpass.getpass("token de GitHub: ")

print("listo para subir:", RUN_ID)

In [ ]:
def git(*args):
    r = subprocess.run(["git"] + list(args), cwd=ROOT, capture_output=True, text=True)
    print((r.stdout + r.stderr).strip())
    return r.returncode


git("add", f"results/runs/{RUN_ID}")
git("commit", "-m", f"exp: {RUN_ID}")

# Traer primero lo que hayan subido los demás. Como cada corrida vive en su propia
# carpeta, no puede haber conflictos de fusión.
git("pull", "--no-rebase", "origin", "main")

# La URL con el token se arma aquí y no se imprime, para que no quede en la salida
# del notebook ni en la configuración del repositorio.
url = REPO.replace("https://", f"https://{TOKEN}@")
r = subprocess.run(["git", "push", url, "main"], cwd=ROOT,
                   capture_output=True, text=True)
print("push correcto" if r.returncode == 0
      else "FALLÓ el push:\n" + r.stderr.replace(TOKEN, "***"))

## 8. Compilar (cuando ya haya varias corridas)

No hace falta ejecutarlo en cada corrida. Reúne todo lo que haya en `results/runs/`,
lo resume en una tabla y **verifica que las corridas sean comparables**: avisa si
detecta semillas distintas, enventanados distintos, señales BOLD que cambiaron entre
corridas o resultados producidos con código sin confirmar.

In [ ]:
!python compile_results.py

## 9. Diagnóstico de orden (opcional, conviene correrlo primero)

¿El orden temporal de las ventanas aporta señal discriminativa? Corre la **misma**
configuración con tres representaciones: `ordered` (secuencia real), `permuted` (ventanas
barajadas dentro de cada sujeto) y `mean_std` (resumen invariante al orden). Si las tres
rinden parecido, el orden no discrimina, y conviene preferir modelos invariantes al orden
—`deepsets`, o `transformer` con `--model-arg positional=false`— frente a las recurrentes.

Requiere un sitio dinámico (no OHSU). Escribe en `results/` como cualquier corrida, así
que `compile_results.py --stats` las compara luego de forma pareada.

In [ ]:
# Diagnóstico de orden: 3 entrenamientos completos con la config actual. Puede tardar.
# La conectividad ordenada se construye una sola vez y se reutiliza para las tres.
assert REPRESENTACION != "static", "El diagnóstico de orden requiere un sitio dinámico (no OHSU)."
for _rep in ["ordered", "permuted", "mean_std"]:
    print(f"\n=== representación: {_rep} ===")
    _rid, _aviso = ejecutar(["--representation", _rep, "--verbose"])
    if _aviso:
        print(_aviso)